# Vietnamese Receipt EDA

## §0 Setup & Data Acquisition

In [ ]:
import os
import random
import hashlib
import json
import yaml
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

# --- Project-local cache dirs (gitignored) ---
PROJECT_ROOT = Path(".").resolve()
DATASETS_DIR = PROJECT_ROOT / "datasets"
os.environ["KAGGLEHUB_CACHE"] = str(DATASETS_DIR / "kagglehub")
os.environ["HF_HOME"] = str(DATASETS_DIR / "huggingface")
(DATASETS_DIR / "kagglehub").mkdir(parents=True, exist_ok=True)
(DATASETS_DIR / "huggingface").mkdir(parents=True, exist_ok=True)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Output dir (recreate from scratch each run) ---
OUT = Path("eda_outputs")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

# --- Download MC-OCR from Kaggle ---
# NOTE: VQA (`5CD-AI/Viet-OCR-VQA-flash2`) is deferred — gated access pending manual review.
import kagglehub
print("Downloading MC-OCR from Kaggle (cached under ./datasets/kagglehub/)...")
mcocr_path = Path(kagglehub.dataset_download("domixi1989/vietnamese-receipts-mc-ocr-2021"))
print(f"MC-OCR unpacked to: {mcocr_path}")

# Reproducibility hash for MC-OCR (Kaggle has no clean revision)
def tree_hash(path):
    h = hashlib.sha256()
    for p in sorted(path.rglob("*")):
        if p.is_file():
            h.update(str(p.relative_to(path)).encode())
            h.update(str(p.stat().st_size).encode())
    return h.hexdigest()
MCOCR_TREE_HASH = tree_hash(mcocr_path)
print(f"MC-OCR tree hash (size-based): {MCOCR_TREE_HASH[:16]}...")

# --- MC-OCR layout ---
# Confirmed via prior inspection of versions/17/ archive:
#   CSV:        mcocr_train_df.csv at the archive root
#   Image dir:  data0.7/data0.7/
#   Filename:   img_id column
mcocr_csv = mcocr_path / "mcocr_train_df.csv"
mcocr_img_dir = mcocr_path / "data0.7" / "data0.7"
print(f"MC-OCR CSV: {mcocr_csv}")
print(f"MC-OCR image dir: {mcocr_img_dir}")
mcocr_df = pd.read_csv(mcocr_csv)
print(f"MC-OCR CSV columns: {list(mcocr_df.columns)}")
print(f"MC-OCR CSV head:\n{mcocr_df.head(2).to_string()}")

def iter_mcocr():
    """Yield (example_id, image_path_str, annotation_dict)."""
    for _, row in mcocr_df.iterrows():
        name = str(row["img_id"])
        img_path = mcocr_img_dir / name
        if not img_path.exists():
            continue
        yield (name, str(img_path), row.to_dict())

# --- Materialize counts (cheap) ---
MCOCR_COUNT = sum(1 for _ in iter_mcocr())
print(f"\nMC-OCR examples reachable: {MCOCR_COUNT}")

## §1 Schema & Field Analysis → schema.json

## §2 Image Characteristics → training_caps.yaml (resolution)

## §3 Text Characteristics → normalization_rules.yaml + training_caps.yaml (seq_length)

## §4 Visual Quality (Automated) → augmentation.yaml

## §5 VQA Structure → prompt_templates.yaml

## §6 Cross-Dataset Comparison → training_strategy.yaml

## §7 Summary & Sanity Checks

In [ ]:
import os, json, yaml
from pathlib import Path

OUT = Path("eda_outputs")

def validate_section_0():
    """§0 produced MC-OCR iterator with non-zero count."""
    assert "MCOCR_COUNT" in globals(), "§0 did not run; MCOCR_COUNT undefined"
    assert MCOCR_COUNT > 0, "MC-OCR yielded zero examples"
    print(f"  §0 OK — MC-OCR: {MCOCR_COUNT}")

def run_all_validations():
    print("Running sanity checks...")
    validate_section_0()
    print("All sanity checks passed.")

run_all_validations()